In [3]:
import pandas as pd

In [5]:
!pip install huggingface_hub[hf_xet]

In [7]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [8]:
!pip install git+https://github.com/tabularis-ai/be_great.git

  Cloning https://github.com/tabularis-ai/be_great.git to c:\users\haava\appdata\local\temp\pip-req-build-wcd88lgu
  Resolved https://github.com/tabularis-ai/be_great.git to commit 468329e6ef34a5908ad639cf4df1ff654a27799e
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/tabularis-ai/be_great.git 'C:\Users\haava\AppData\Local\Temp\pip-req-build-wcd88lgu'


In [12]:
df = pd.read_csv(r"C:\Users\haava\Documents\student\student-mat.csv", sep=";")

In [13]:
drop_cols = [
    "G1",
    "G2",
    "guardian",
    "reason",
    "nursery",
    "romantic",
    "schoolsup"
]

df = df.drop(columns=drop_cols)

# clean strings for generation
for c in df.select_dtypes(include="object"):
    df[c] = (
        df[c]
        .astype(str)
        .str.replace(",", " ", regex=False)
        .str.replace("\n", " ", regex=False)
        .str.strip()
    )

df = df.fillna("")

print(df.shape)   # should be ~ (395, 26 or 27)

(395, 26)


In [14]:
categorical = df.select_dtypes(include="object").columns
numeric = df.select_dtypes(exclude="object").columns

df = df[list(categorical) + list(numeric)]

In [15]:
from sklearn.model_selection import train_test_split

D_real_train, D_real_holdout = train_test_split(
    df,
    train_size=300,
    test_size=95,
    random_state=42,
    shuffle=True
)

In [16]:
df.head().transpose()

,0,1,2,3,4
school,GP,GP,GP,GP,GP
sex,F,F,F,F,F
address,U,U,U,U,U
famsize,GT3,GT3,LE3,GT3,GT3
Pstatus,A,T,T,T,T
Mjob,at_home,at_home,at_home,health,other
Fjob,teacher,other,other,services,other
famsup,no,yes,no,yes,yes
paid,no,no,yes,yes,yes
activities,no,no,no,yes,no


In [23]:
D_real_train.shape

(300, 26)

In [25]:
print(df.columns)

Index(['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob',
       'famsup', 'paid', 'activities', 'higher', 'internet', 'age', 'Medu',
       'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime',
       'goout', 'Dalc', 'Walc', 'health', 'absences', 'G3'],
      dtype='object')


In [34]:
from be_great import GReaT

model = GReaT(
    llm="distilgpt2",
    batch_size=8,
    epochs=40,


    fp16=True,
)

model.fit(D_real_train)

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
500,1.183200
1000,0.835200
1500,0.796100


In [38]:

synthetic = model.sample(
    n_samples=300,
    max_length=200,
    temperature=0.7,
    guided_sampling = False
)

print("Generated:", len(synthetic))
synthetic.to_csv("synthetic_students.csv", index=False)

323it [00:19, 16.66it/s]                                                                                               

Generated: 300


In [40]:
df_syn = pd.read_csv("synthetic_students.csv", sep=",")

In [42]:
df_syn.shape


(300, 26)

In [44]:
df_syn.head().transpose()

,0,1,2,3,4
school,MS,GP,GP,GP,GP
sex,M,F,M,M,M
address,U,R,U,U,U
famsize,GT3,GT3,LE3,GT3,GT3
Pstatus,T,T,T,T,T
Mjob,at_home,at_home,at_home,at_home,at_home
Fjob,other,services,other,services,at_home
famsup,no,no,yes,no,yes
paid,no,no,no,yes,no
activities,no,no,no,yes,no


In [46]:
df.head().transpose()

,0,1,2,3,4
school,GP,GP,GP,GP,GP
sex,F,F,F,F,F
address,U,U,U,U,U
famsize,GT3,GT3,LE3,GT3,GT3
Pstatus,A,T,T,T,T
Mjob,at_home,at_home,at_home,health,other
Fjob,teacher,other,other,services,other
famsup,no,yes,no,yes,yes
paid,no,no,yes,yes,yes
activities,no,no,no,yes,no


In [48]:
def cast_synthetic_to_original_schema(synth: pd.DataFrame, original: pd.DataFrame) -> pd.DataFrame:
    synth = synth.copy()

    synth = synth[[c for c in original.columns if c in synth.columns]]

    for col in synth.columns:
        orig_col = original[col]


        if pd.api.types.is_object_dtype(orig_col) or pd.api.types.is_string_dtype(orig_col):
            synth[col] = synth[col].astype(str)

        elif pd.api.types.is_integer_dtype(orig_col):
            synth[col] = pd.to_numeric(synth[col], errors="coerce")
         
            fill_value = int(orig_col.median())
            synth[col] = synth[col].fillna(fill_value).round().astype(orig_col.dtype)

        elif pd.api.types.is_float_dtype(orig_col):
            synth[col] = pd.to_numeric(synth[col], errors="coerce")

            fill_value = float(orig_col.median())
            synth[col] = synth[col].fillna(fill_value).astype(orig_col.dtype)
 
        elif pd.api.types.is_bool_dtype(orig_col):
            mapping = {
                "true": True, "false": False,
                "1": True, "0": False,
                "yes": True, "no": False
            }
            synth[col] = (
                synth[col]
                .astype(str)
                .str.strip()
                .str.lower()
                .map(mapping)
                .fillna(False)
                .astype(bool)
            )

        else:

            synth[col] = synth[col].astype(orig_col.dtype, errors="ignore")

    return synth

synthetic_casted = cast_synthetic_to_original_schema(df_syn, df)

print(synthetic_casted.dtypes)
synthetic_casted.to_csv("synthetic_students_typed.csv", index=False)

school        object
sex           object
address       object
famsize       object
Pstatus       object
Mjob          object
Fjob          object
famsup        object
paid          object
activities    object
higher        object
internet      object
age            int64
Medu           int64
Fedu           int64
traveltime     int64
studytime      int64
failures       int64
famrel         int64
freetime       int64
goout          int64
Dalc           int64
Walc           int64
health         int64
absences       int64
G3             int64
dtype: object


In [50]:
df_syn_typed = pd.read_csv("synthetic_students_typed.csv", sep=",")

In [54]:
df_syn_typed.head()

,school,sex,address,famsize,Pstatus,Mjob,Fjob,famsup,paid,activities,...,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G3
0,MS,M,U,GT3,T,at_home,other,no,no,no,...,2,0,5,2,3,1,1,3,4,10
1,GP,F,R,GT3,T,at_home,services,no,no,no,...,2,0,4,3,5,1,1,2,0,10
2,GP,M,U,LE3,T,at_home,other,yes,no,no,...,1,0,4,3,3,1,1,1,2,14
3,GP,M,U,GT3,T,at_home,services,no,yes,yes,...,3,0,3,4,2,1,1,5,14,14
4,GP,M,U,GT3,T,at_home,at_home,yes,no,no,...,3,0,4,3,3,1,1,5,0,10
